In [ ]:
#S4M-0 — Setup
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, shutil, subprocess, sys, platform, time
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd

BASE        = '/content/drive/MyDrive/BEM-LLM'
DATA        = f'{BASE}/data'
REF         = f'{DATA}/idf_referans'
WEATHER_DIR = f'{DATA}/epw_iklim'
ANALIZ      = f'{BASE}/analiz'
SPRINT2_DIR = f'{ANALIZ}/sprint2_lobo'
PROMPTS_DIR = f'{ANALIZ}/sprint3_prompts'
SIM_DIR     = f'{ANALIZ}/sprint4_simulation'
IDF_OUT     = f'{BASE}/idf_llm'
EP_OUT      = f'{BASE}/sonuclar'
os.makedirs(SIM_DIR, exist_ok=True)
os.makedirs(IDF_OUT, exist_ok=True)
os.makedirs(EP_OUT, exist_ok=True)

EP = '/usr/local/EnergyPlus/energyplus'
EP_TIMEOUT_S = 1800
EP_VERSION_TAG = '22.1.0-ed759b17ee'
EP_DOWNLOAD_URL = ('https://github.com/NREL/EnergyPlus/releases/download/v22.1.0/'
                    'EnergyPlus-22.1.0-ed759b17ee-Linux-Ubuntu20.04-x86_64.tar.gz')

installed_version = None
if os.path.exists(EP):
    installed_version = subprocess.run([EP, '--version'], capture_output=True, text=True).stdout.strip()

if not os.path.exists(EP) or EP_VERSION_TAG not in (installed_version or ''):
    if os.path.exists('/usr/local/EnergyPlus'):
        subprocess.run(['rm', '-rf', '/usr/local/EnergyPlus'])
    subprocess.run(['wget', '-q', '-O', '/tmp/ep.tar.gz', EP_DOWNLOAD_URL])
    os.makedirs('/usr/local/EnergyPlus', exist_ok=True)
    subprocess.run(['tar', '-xzf', '/tmp/ep.tar.gz', '-C', '/usr/local/EnergyPlus', '--strip-components=1'])

energyplus_version = subprocess.run([EP, '--version'], capture_output=True, text=True).stdout.strip()
print(energyplus_version)
assert '22.1' in energyplus_version, f'Expected EnergyPlus 22.1.x, got: {energyplus_version}'

with open(f'{ANALIZ}/protocol.json') as f:
    protocol = json.load(f)
with open(f'{SPRINT2_DIR}/lobo_range_table.json') as f:
    lobo = json.load(f)
with open(f'{SPRINT2_DIR}/control_parameters.json') as f:
    control_data = json.load(f)

TESTED_BUILDINGS = protocol['tested_buildings']
MODELS   = protocol['models']
SEEDS    = protocol['seeds']
CLIMATES = protocol['climates']
TARGET_PARAMS  = lobo['target_parameters']
CONTROL_PARAMS = lobo['control_parameters']
CONTROL_VALUES = control_data['values_by_building']

CLIMATE_EPW = {'5A': f'{WEATHER_DIR}/5A_Buffalo.epw'}

TOLERANCE_PCT = {
    'window_u_value': 0.5, 'window_shgc': 0.5, 'lighting_W_m2': 0.5,
    'equipment_W_m2': 1.0, 'occupancy_m2_person': 1.0, 'outdoor_air': 2.0,
    'cooling_cop': 1.0, 'heating_setpoint_C': 0.5, 'cooling_setpoint_C': 0.5,
}

N_WORKERS = multiprocessing.cpu_count()
print(f'{len(TESTED_BUILDINGS)} buildings, {len(MODELS)} models, {len(SEEDS)} seeds, '
      f'{multiprocessing.cpu_count()} CPUs ({N_WORKERS} workers)')

Mounted at /content/drive
EnergyPlus, Version 22.1.0-ed759b17ee, YMD=2026.07.29 05:55
5 buildings, 2 models, 5 seeds, 2 CPUs (2 workers)


In [ ]:
#S4M-A — Extraction
def load_idf_text(path):
    with open(path, 'r', errors='ignore') as f:
        return f.read()


def safe_float(v):
    try:
        f = float(v)
        return f if np.isfinite(f) else None
    except (TypeError, ValueError):
        return None


def weighted_mean(values, weights):
    values, weights = np.array(values, dtype=float), np.array(weights, dtype=float)
    return float(np.average(values, weights=weights)) if len(values) else None


def iter_blocks(idf_text):
    parts = re.split(r'(;[ \t]*(?:!-[^\n]*)?\n)', idf_text)
    blocks = [''.join(parts[i:i + 2]) for i in range(0, len(parts) - 1, 2)]
    if len(parts) % 2 == 1:
        blocks.append(parts[-1])
    for block in blocks:
        stripped = block.strip()
        if not stripped:
            continue
        obj_type = None
        for line in stripped.split('\n'):
            line_s = line.strip()
            if not line_s or line_s.startswith('!'):
                continue
            obj_type = line_s.split(',')[0].strip().upper()
            break
        if obj_type:
            yield obj_type, block


def blocks_of(idf_text, object_type):
    return [b for t, b in iter_blocks(idf_text) if t == object_type.upper()]


def value_before_comment(block, comment_substring):
    m = re.search(rf'([\w.+\-]+)\s*[,;]\s*!-[^\n]*{re.escape(comment_substring)}', block, re.IGNORECASE)
    return safe_float(m.group(1)) if m else None


def text_before_comment(block, comment_substring):
    m = re.search(rf'([^\n,;]+)\s*[,;]\s*!-[^\n]*{re.escape(comment_substring)}', block, re.IGNORECASE)
    return m.group(1).strip() if m else None


def polygon_area(block):
    coords = re.findall(r'([\-\d.]+)\s*,\s*([\-\d.]+)\s*,\s*([\-\d.]+)\s*[,;]?\s*!-[^\n]*Vertex', block)
    if len(coords) < 3:
        return None
    pts = [np.array([float(x), float(y), float(z)]) for x, y, z in coords]
    normal = np.zeros(3)
    for i in range(len(pts)):
        normal += np.cross(pts[i], pts[(i + 1) % len(pts)])
    return float(np.linalg.norm(normal)) / 2.0


def zone_area_map(idf_text):
    areas = {}
    for block in blocks_of(idf_text, 'BuildingSurface:Detailed'):
        if (text_before_comment(block, 'Surface Type') or '').strip().lower() != 'floor':
            continue
        zone = text_before_comment(block, 'Zone Name')
        area = polygon_area(block)
        if zone and area:
            areas[zone.strip().upper()] = areas.get(zone.strip().upper(), 0.0) + area
    return areas


def get_window_u_shgc(idf_text):
    con_glazing = {}
    for block in blocks_of(idf_text, 'Construction'):
        name = text_before_comment(block, 'Name')
        outside_layer = text_before_comment(block, 'Outside Layer')
        if name and outside_layer:
            con_glazing[name.upper()] = outside_layer.rstrip(';').upper()

    glazing = {}
    for block in blocks_of(idf_text, 'WindowMaterial:SimpleGlazingSystem'):
        name = text_before_comment(block, 'Name')
        u = value_before_comment(block, 'U-Factor')
        shgc = value_before_comment(block, 'Solar Heat Gain Coefficient')
        if name and u and shgc:
            glazing[name.upper()] = (u, shgc)

    us, shgcs, areas = [], [], []
    for block in blocks_of(idf_text, 'FenestrationSurface:Detailed'):
        if (text_before_comment(block, 'Surface Type') or '').strip().lower() != 'window':
            continue
        con_name = text_before_comment(block, 'Construction Name')
        props = glazing.get(con_glazing.get((con_name or '').upper(), '').upper())
        if not props:
            continue
        area = polygon_area(block) or 1.0
        us.append(props[0]); shgcs.append(props[1]); areas.append(area)
    if not us:
        return None, None
    return round(weighted_mean(us, areas), 4), round(weighted_mean(shgcs, areas), 4)


def get_lighting(idf_text, zmap):
    vals, wts = [], []
    for block in blocks_of(idf_text, 'Lights'):
        v = value_before_comment(block, 'Watts per Zone Floor Area')
        if v:
            zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
            vals.append(v); wts.append(zmap.get(zone, 1.0))
    return round(weighted_mean(vals, wts), 3) if vals else None


EXCLUDE_ZONES = ['CORRIDOR', 'MECH', 'BATH', 'LOBBY', 'STAIR', 'PLENUM', 'JANITOR']


def get_equipment(idf_text, zmap, building_name=None):
    exclude = EXCLUDE_ZONES + (['STORAGE'] if building_name != 'Warehouse' else [])
    vals, wts = [], []
    for block in blocks_of(idf_text, 'ElectricEquipment'):
        zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
        if any(kw in zone for kw in exclude):
            continue
        area = zmap.get(zone)
        wa = value_before_comment(block, 'Watts per Zone Floor Area')
        if wa:
            vals.append(wa); wts.append(area or 1.0)
            continue
        design_w = value_before_comment(block, 'Design Level {W}')
        if design_w and area:
            density = design_w / area
            if density <= 150.0:
                vals.append(density); wts.append(area)
    return round(weighted_mean(vals, wts), 3) if vals else None


def get_occupancy(idf_text, zmap):
    vals, wts = [], []
    for block in blocks_of(idf_text, 'People'):
        zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
        area = zmap.get(zone)
        apf = value_before_comment(block, 'Floor Area per Person')
        if apf:
            vals.append(apf); wts.append(area or 1.0)
            continue
        ppa = value_before_comment(block, 'People per Floor Area')
        if ppa:
            vals.append(1.0 / ppa); wts.append(area or 1.0)
            continue
        m = re.search(r'([\w.+\-]+)\s*,\s*!-\s*Number of People\s*\n', block, re.IGNORECASE)
        n = safe_float(m.group(1)) if m else None
        if n and area:
            vals.append(area / n); wts.append(area)
    return round(weighted_mean(vals, wts), 3) if vals else None


def get_outdoor_air(idf_text, occupancy_density):
    vals, wts = [], []
    for block in blocks_of(idf_text, 'DesignSpecification:OutdoorAir'):
        method = (text_before_comment(block, 'Outdoor Air Method') or '').strip().lower()
        per_area = value_before_comment(block, 'Outdoor Air Flow per Zone Floor Area') or 0.0
        per_person_area = (value_before_comment(block, 'Outdoor Air Flow per Person') or 0.0) / (occupancy_density or 18.0)
        combined = max(per_area, per_person_area) if method == 'maximum' else per_area + per_person_area
        if combined > 0:
            vals.append(combined); wts.append(1.0)
    return round(weighted_mean(vals, wts), 6) if vals else None


def get_cooling_cop(idf_text):
    vals, wts = [], []
    for obj_type in ['Coil:Cooling:DX:SingleSpeed', 'Coil:Cooling:DX:TwoSpeed', 'Coil:Cooling:DX:MultiSpeed']:
        for block in blocks_of(idf_text, obj_type):
            caps = [safe_float(v) for v in re.findall(r'([\d.]+)\s*,\s*!-[^\n]*Rated Total Cooling Capacity', block)]
            cops = [safe_float(v) for v in re.findall(r'([\d.]+)\s*,\s*!-[^\n]*Rated Cooling COP', block)]
            caps = [c for c in caps if c is not None]
            cops = [c for c in cops if c is not None]
            for i, cop in enumerate(cops):
                vals.append(cop); wts.append(caps[i] if i < len(caps) else 1.0)
    return round(weighted_mean(vals, wts), 4) if vals else None


def get_setpoints(idf_text):
    sched_vals = {}
    for block in blocks_of(idf_text, 'Schedule:Compact'):
        name = text_before_comment(block, 'Name')
        vals = [safe_float(v) for v in re.findall(r'Until:\s*\d{1,2}:\d{2}\s*,\s*([\d.\-]+)', block)]
        vals = [v for v in vals if v is not None]
        if name and vals:
            sched_vals[name.upper()] = vals

    heat_vals, cool_vals = [], []
    for block in blocks_of(idf_text, 'ThermostatSetpoint:DualSetpoint'):
        heat_sched = text_before_comment(block, 'Heating Setpoint Temperature Schedule Name')
        cool_sched = text_before_comment(block, 'Cooling Setpoint Temperature Schedule Name')
        h = [v for v in sched_vals.get((heat_sched or '').upper(), []) if 18.0 <= v <= 26.0]
        c = [v for v in sched_vals.get((cool_sched or '').upper(), []) if 20.0 <= v <= 27.0]
        if h:
            heat_vals.append(float(np.median(h)))
        if c:
            cool_vals.append(float(np.median(c)))
    heat_sp = round(float(np.mean(heat_vals)), 2) if heat_vals else None
    cool_sp = round(float(np.mean(cool_vals)), 2) if cool_vals else None
    return heat_sp, cool_sp


def extract_all_params(idf_text, building_name=None):
    zmap = zone_area_map(idf_text)
    u, shgc = get_window_u_shgc(idf_text)
    occ = get_occupancy(idf_text, zmap)
    heat_sp, cool_sp = get_setpoints(idf_text)
    return {
        'window_u_value': u, 'window_shgc': shgc,
        'lighting_W_m2': get_lighting(idf_text, zmap),
        'equipment_W_m2': get_equipment(idf_text, zmap, building_name),
        'occupancy_m2_person': occ,
        'outdoor_air': get_outdoor_air(idf_text, occ),
        'cooling_cop': get_cooling_cop(idf_text),
        'heating_setpoint_C': heat_sp, 'cooling_setpoint_C': cool_sp,
    }

In [ ]:
#S4M-B — Injection
def _replace_before_comment(block, comment_substring, new_value):
    pattern = re.compile(rf'([\w.+\-]*)(\s*[,;]\s*!-[^\n]*{re.escape(comment_substring)})', re.IGNORECASE)
    return pattern.sub(lambda m: f'{new_value}{m.group(2)}', block, count=1)


def _rebuild(idf_text, object_type, transform):
    out = []
    for otype, block in iter_blocks(idf_text):
        out.append(transform(block) if otype == object_type.upper() else block)
    return ''.join(out)


def inject_window_u_shgc(idf_text, u_value, shgc):
    def fix(block):
        block = _replace_before_comment(block, 'U-Factor', round(u_value, 4))
        return _replace_before_comment(block, 'Solar Heat Gain Coefficient', round(shgc, 4))
    return _rebuild(idf_text, 'WindowMaterial:SimpleGlazingSystem', fix)


def inject_lights(idf_text, value_w_m2):
    def fix(block):
        if value_before_comment(block, 'Watts per Zone Floor Area') is not None:
            return _replace_before_comment(block, 'Watts per Zone Floor Area', round(value_w_m2, 3))
        return block
    return _rebuild(idf_text, 'Lights', fix)


def inject_equipment(idf_text, value_w_m2, zmap):
    def fix(block):
        if value_before_comment(block, 'Watts per Zone Floor Area') is not None:
            return _replace_before_comment(block, 'Watts per Zone Floor Area', round(value_w_m2, 3))
        if value_before_comment(block, 'Design Level {W}') is not None:
            zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
            area = zmap.get(zone)
            if area:
                return _replace_before_comment(block, 'Design Level {W}', round(value_w_m2 * area, 2))
        return block
    return _rebuild(idf_text, 'ElectricEquipment', fix)


def inject_occupancy(idf_text, value_m2_person, zmap):
    def fix(block):
        if value_before_comment(block, 'Floor Area per Person') is not None:
            return _replace_before_comment(block, 'Floor Area per Person', round(value_m2_person, 3))
        if value_before_comment(block, 'People per Floor Area') is not None:
            return _replace_before_comment(block, 'People per Floor Area', round(1.0 / value_m2_person, 6))
        m = re.search(r'([\w.+\-]+)(\s*,\s*!-\s*Number of People\s*\n)', block, re.IGNORECASE)
        if m and safe_float(m.group(1)) is not None:
            zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
            area = zmap.get(zone)
            if area:
                new_n = round(area / value_m2_person, 3)
                return block[:m.start(1)] + str(new_n) + block[m.end(1):]
        return block
    return _rebuild(idf_text, 'People', fix)


def inject_outdoor_air(idf_text, value_m3_s_m2):
    def fix(block):
        return _replace_before_comment(block, 'Outdoor Air Flow per Zone Floor Area', round(value_m3_s_m2, 6))
    return _rebuild(idf_text, 'DesignSpecification:OutdoorAir', fix)


def inject_cooling_cop(idf_text, cop):
    pattern = re.compile(r'([\d.]+)(\s*,\s*!-[^\n]*Rated Cooling COP)', re.IGNORECASE)
    return pattern.sub(lambda m: f'{round(cop, 3)}{m.group(2)}', idf_text)


def inject_setpoint_schedule(idf_text, schedule_name, value_c):
    block_pattern = re.compile(rf'(Schedule:Compact\s*,\s*{re.escape(schedule_name)}\s*,.*?;)',
                               re.IGNORECASE | re.DOTALL)
    match = block_pattern.search(idf_text)
    if not match:
        return idf_text
    new_block = re.sub(r'(Until:\s*\d{1,2}:\d{2}\s*,)\s*[\d.\-]+', rf'\g<1>{round(value_c, 2)}', match.group(1))
    return idf_text.replace(match.group(1), new_block)


def inject_setpoints(idf_text, heating_c, cooling_c):
    for block in blocks_of(idf_text, 'ThermostatSetpoint:DualSetpoint'):
        heat_sched = text_before_comment(block, 'Heating Setpoint Temperature Schedule Name')
        cool_sched = text_before_comment(block, 'Cooling Setpoint Temperature Schedule Name')
        if heat_sched:
            idf_text = inject_setpoint_schedule(idf_text, heat_sched, heating_c)
        if cool_sched:
            idf_text = inject_setpoint_schedule(idf_text, cool_sched, cooling_c)
    return idf_text


def inject_all_parameters(idf_path, out_path, values):
    text = load_idf_text(idf_path)
    zmap = zone_area_map(text)
    text = inject_window_u_shgc(text, values['window_u_value'], values['window_shgc'])
    text = inject_lights(text, values['lighting_W_m2'])
    text = inject_equipment(text, values['equipment_W_m2'], zmap)
    text = inject_occupancy(text, values['occupancy_m2_person'], zmap)
    text = inject_outdoor_air(text, values['outdoor_air'])
    text = inject_cooling_cop(text, values['cooling_cop'])
    text = inject_setpoints(text, values['heating_setpoint_C'], values['cooling_setpoint_C'])
    with open(out_path, 'w') as f:
        f.write(text)

In [ ]:
#S4M-C — Standard output objects
STANDARD_OUTPUTS = """
Output:Meter,Electricity:Facility,Monthly;
Output:Meter,NaturalGas:Facility,Monthly;
Output:Meter,DistrictHeating:Facility,Monthly;
Output:Meter,DistrictCooling:Facility,Monthly;
Output:Variable,*,Zone Air System Sensible Heating Energy,Monthly;
Output:Variable,*,Zone Air System Sensible Cooling Energy,Monthly;
"""

def _strip_idf_comments(idf_text):
    return re.sub(r'!.*', '', idf_text)

def ensure_standard_outputs(idf_path):
    text = load_idf_text(idf_path)
    active_text = _strip_idf_comments(text)
    if 'Output:Meter,Electricity:Facility' not in active_text:
        with open(idf_path, 'w') as f:
            f.write(text.rstrip() + '\n' + STANDARD_OUTPUTS)

In [ ]:
#S4M-D — Round-trip verification + EnergyPlus runner
def round_trip_check(idf_path, target_values, building_name=None):
    readback = extract_all_params(load_idf_text(idf_path), building_name)
    report, passed = [], True
    for param, target in target_values.items():
        actual = readback.get(param)
        tol = TOLERANCE_PCT.get(param, 0.5)
        if actual is None:
            report.append({'parameter': param, 'target': target, 'readback': None, 'status': 'MISSING'})
            passed = False
            continue
        diff_pct = abs(actual - target) / abs(target) * 100 if target else abs(actual - target) * 100
        ok = diff_pct <= tol
        passed = passed and ok
        report.append({'parameter': param, 'target': round(target, 6), 'readback': round(actual, 6),
                        'diff_pct': round(diff_pct, 4), 'tolerance_pct': tol,
                        'status': 'OK' if ok else 'TOLERANCE_EXCEEDED'})
    return passed, report


def build_and_check(building, values, tag):
    out_idf = f'{IDF_OUT}/{tag}.idf'
    shutil.copy(f'{REF}/ASHRAE901_{building}_STD2019_Buffalo.idf', out_idf)
    inject_all_parameters(out_idf, out_idf, values)
    ensure_standard_outputs(out_idf)
    return out_idf, round_trip_check(out_idf, values, building_name=building)


def run_energyplus(idf_path, epw_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    try:
        result = subprocess.run([EP, '-w', epw_path, '-d', out_dir, '-r', idf_path],
                                capture_output=True, text=True, timeout=EP_TIMEOUT_S)
        return result.returncode == 0, result.stderr
    except subprocess.TimeoutExpired:
        return False, f'TIMEOUT: exceeded {EP_TIMEOUT_S}s'

In [ ]:
#S4M-E — Simulation vector manifest
param_debug = pd.read_csv(f'{PROMPTS_DIR}/param_debug_mapping.csv')
ep_baselines = pd.read_csv(f'{SPRINT2_DIR}/energyplus_baseline_vectors.csv')
range_table = lobo['range_table']

sim_vectors = []

llm_main = param_debug[param_debug['climate'] == CLIMATES['main']]
for (building, fmt, model, seed), group in llm_main.groupby(['building', 'format', 'model', 'seed']):
    values = dict(zip(group['true_name'], group['value']))
    if set(TARGET_PARAMS).issubset(values):
        sim_vectors.append({'building': building, 'climate': CLIMATES['main'],
                             'source': f'{fmt}_{model.split("/")[-1]}_seed{seed}', 'source_type': 'llm',
                             'format': fmt, 'model': model, 'seed': seed,
                             **{p: values[p] for p in TARGET_PARAMS}})

for _, row in ep_baselines.iterrows():
    sim_vectors.append({'building': row['building'], 'climate': CLIMATES['main'],
                         'source': row['baseline_type'], 'source_type': 'deterministic_baseline',
                         'format': None, 'model': None, 'seed': None,
                         **{p: row[p] for p in TARGET_PARAMS}})

N_RANDOM_VECTORS = 5
rng = np.random.default_rng(protocol['master_seed'])
for building in TESTED_BUILDINGS:
    for i in range(N_RANDOM_VECTORS):
        values = {p: rng.uniform(*range_table[building][p]) for p in TARGET_PARAMS}
        sim_vectors.append({'building': building, 'climate': CLIMATES['main'],
                             'source': f'uniform_random_{i}', 'source_type': 'uniform_random',
                             'format': None, 'model': None, 'seed': None, **values})

for building in TESTED_BUILDINGS:
    ref_values = extract_all_params(load_idf_text(f'{REF}/ASHRAE901_{building}_STD2019_Buffalo.idf'), building)
    sim_vectors.append({'building': building, 'climate': CLIMATES['main'],
                         'source': 'DOE_reference', 'source_type': 'doe_reference',
                         'format': None, 'model': None, 'seed': None, **ref_values})

sim_manifest_df = pd.DataFrame(sim_vectors)
sim_manifest_df.to_csv(f'{SIM_DIR}/simulation_vector_manifest.csv', index=False)
print(f'{len(sim_manifest_df)} vectors registered (5A main).')
print(sim_manifest_df.groupby('source_type').size().to_string())

240 vectors registered (5A main).
source_type
deterministic_baseline     10
doe_reference               5
llm                       200
uniform_random             25


In [ ]:
#S4M-F — Smoke test
def smoke_test_vector(row, epw_path, index, total):
    tag = f'{row["building"]}_{row["climate"]}_{row["source"]}'
    print(f'[{index}/{total}] {tag} ...', end=' ', flush=True)
    t0 = time.time()

    values = {p: row[p] for p in TARGET_PARAMS}
    out_idf, (rt_passed, rt_report) = build_and_check(row['building'], values, tag)
    if not rt_passed:
        print(f'ROUND-TRIP FAILED ({time.time() - t0:.1f}s)')
        return {'tag': tag, 'round_trip_ok': False, 'energyplus_ok': None, 'report': rt_report}

    ep_ok, stderr = run_energyplus(out_idf, epw_path, f'{EP_OUT}/{tag}')
    status = 'OK' if ep_ok else ('TIMEOUT' if 'TIMEOUT' in str(stderr) else 'EP FAILED')
    print(f'{status} ({time.time() - t0:.1f}s)')
    return {'tag': tag, 'round_trip_ok': True, 'energyplus_ok': ep_ok,
            'stderr_tail': stderr[-500:] if not ep_ok else None}


epw_5A = CLIMATE_EPW['5A']

doe_rows = [{'building': b, 'climate': CLIMATES['main'], 'source': 'DOE_reference',
             **extract_all_params(load_idf_text(f'{REF}/ASHRAE901_{b}_STD2019_Buffalo.idf'), b)}
            for b in TESTED_BUILDINGS]
doe_df = pd.DataFrame(doe_rows)

per_format_model = sim_manifest_df[sim_manifest_df.source_type == 'llm'].groupby(['format', 'model']).head(1)
per_baseline = sim_manifest_df[sim_manifest_df.source_type == 'deterministic_baseline'].groupby(['building', 'source']).head(1)
per_random = sim_manifest_df[sim_manifest_df.source_type == 'uniform_random'].groupby('building').head(1)

smoke_sample = pd.concat([doe_df, per_format_model, per_baseline, per_random], ignore_index=True)
print(f'Smoke sample: {len(smoke_sample)} cases\n')

batch_start = time.time()
smoke_results = []
for i, (_, row) in enumerate(smoke_sample.iterrows(), start=1):
    smoke_results.append(smoke_test_vector(row, epw_5A, i, len(smoke_sample)))
    pd.DataFrame(smoke_results).to_csv(f'{SIM_DIR}/smoke_test_results_main.csv', index=False)
    elapsed = time.time() - batch_start
    print(f'    -> avg {elapsed/i:.1f}s/case, est. remaining {elapsed/i*(len(smoke_sample)-i)/60:.1f} min')

smoke_df = pd.DataFrame(smoke_results)
rt_rejected = (~smoke_df['round_trip_ok']).sum()
ep_eligible = smoke_df[smoke_df['round_trip_ok']]
smoke_passed = bool(ep_eligible['energyplus_ok'].fillna(False).infer_objects(copy=False).all())

print(f'\nSmoke test: {"PASSED" if smoke_passed else "FAILED"} ({len(smoke_df)} cases, '
      f'{rt_rejected} round-trip rejection(s) -- expected, not a failure)')
if rt_rejected:
    print(smoke_df[~smoke_df['round_trip_ok']][['tag']].to_string(index=False))
if not smoke_passed:
    print(ep_eligible[~ep_eligible['energyplus_ok'].fillna(False)].to_string())

Smoke sample: 28 cases

[1/28] OfficeMedium_5A_DOE_reference ... OK (113.6s)
    -> avg 113.6s/case, est. remaining 51.1 min
[2/28] SchoolPrimary_5A_DOE_reference ... OK (245.3s)
    -> avg 179.4s/case, est. remaining 77.8 min
[3/28] RetailStripmall_5A_DOE_reference ... OK (84.1s)
    -> avg 147.7s/case, est. remaining 61.5 min
[4/28] ApartmentMidRise_5A_DOE_reference ... OK (324.2s)
    -> avg 191.8s/case, est. remaining 76.7 min
[5/28] RestaurantFastFood_5A_DOE_reference ... OK (65.8s)
    -> avg 166.6s/case, est. remaining 63.9 min
[6/28] ApartmentMidRise_5A_F1_llama-3.3-70b-versatile_seed1542799867 ... ROUND-TRIP FAILED (0.6s)
    -> avg 138.9s/case, est. remaining 50.9 min
[7/28] ApartmentMidRise_5A_F1_gpt-oss-120b_seed1542799867 ... OK (244.9s)
    -> avg 154.1s/case, est. remaining 53.9 min
[8/28] ApartmentMidRise_5A_F2_llama-3.3-70b-versatile_seed1542799867 ... OK (357.4s)
    -> avg 179.5s/case, est. remaining 59.8 min
[9/28] ApartmentMidRise_5A_F2_gpt-oss-120b_seed1542799867 

/tmp/ipykernel_834/1249068832.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  smoke_passed = bool(ep_eligible['energyplus_ok'].fillna(False).infer_objects(copy=False).all())


In [ ]:
#S4M-G — Full batch
assert smoke_passed, 'Smoke test failed -- inspect smoke_test_results_main.csv before proceeding.'

def _run_one_case(args):
    building, climate, source, fmt, model, seed, values, epw_path = args
    tag = f'{building}_{climate}_{source}'
    t0 = time.time()
    out_idf, (rt_passed, rt_report) = build_and_check(building, values, tag)
    if not rt_passed:
        return {'tag': tag, 'building': building, 'climate': climate, 'source': source,
                'format': fmt, 'model': model, 'seed': seed,
                'round_trip_ok': False, 'energyplus_ok': None,
                'duration_s': round(time.time() - t0, 1), 'rt_report': rt_report}
    ep_ok, _ = run_energyplus(out_idf, epw_path, f'{EP_OUT}/{tag}')
    return {'tag': tag, 'building': building, 'climate': climate, 'source': source,
            'format': fmt, 'model': model, 'seed': seed,
            'round_trip_ok': True, 'energyplus_ok': ep_ok,
            'duration_s': round(time.time() - t0, 1), 'rt_report': rt_report}


def _flatten_report(result):
    return [{'tag': result['tag'], 'building': result['building'], 'climate': result['climate'],
             'source': result['source'], 'format': result['format'], 'model': result['model'],
             'seed': result['seed'], 'duration_s': result['duration_s'], **p}
            for p in (result.get('rt_report') or [])]


def run_batch_parallel(manifest_df, epw_path, log_name, adherence_name):
    log_path = f'{SIM_DIR}/{log_name}'
    adherence_path = f'{SIM_DIR}/{adherence_name}'

    log = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []
    adherence_rows = pd.read_csv(adherence_path).to_dict('records') if os.path.exists(adherence_path) else []
    done_tags = {r['tag'] for r in log if r.get('energyplus_ok') is not None or r.get('round_trip_ok') is False}

    jobs = []
    for _, row in manifest_df.iterrows():
        tag = f'{row["building"]}_{row["climate"]}_{row["source"]}'
        if tag in done_tags:
            continue
        values = {p: row[p] for p in TARGET_PARAMS}
        jobs.append((row['building'], row['climate'], row['source'],
                     row.get('format'), row.get('model'), row.get('seed'), values, epw_path))

    print(f'{len(jobs)} case(s) remaining (of {len(manifest_df)} total).')
    completed = 0
    batch_start = time.time()
    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(_run_one_case, job): job for job in jobs}
        for future in as_completed(futures):
            result = future.result()
            log.append({k: v for k, v in result.items() if k != 'rt_report'})
            adherence_rows.extend(_flatten_report(result))
            completed += 1
            status = ('OK' if result.get('energyplus_ok')
                       else 'ROUND-TRIP FAILED' if not result['round_trip_ok'] else 'EP FAILED')
            elapsed = time.time() - batch_start
            print(f'  [{completed}/{len(jobs)}] {result["tag"]}: {status} ({result["duration_s"]:.0f}s)  '
                  f'(avg {elapsed/completed:.0f}s/case, est. remaining {elapsed/completed*(len(jobs)-completed)/3600:.1f}h)')
            if completed % 10 == 0:
                pd.DataFrame(log).to_csv(log_path, index=False)
                pd.DataFrame(adherence_rows).to_csv(adherence_path, index=False)

    pd.DataFrame(log).to_csv(log_path, index=False)
    pd.DataFrame(adherence_rows).to_csv(adherence_path, index=False)
    return pd.DataFrame(log)

main_batch_df = run_batch_parallel(sim_manifest_df, epw_5A, 'full_batch_log_5A.csv', 'range_adherence_log_main.csv')
print(f'\n5A main batch: {main_batch_df.energyplus_ok.sum()}/{len(main_batch_df)} successful.')
print(f'Mean duration: {main_batch_df.duration_s.mean():.1f}s/case, '
      f'total: {main_batch_df.duration_s.sum()/3600:.2f}h (sequential-equivalent)')

240 case(s) remaining (of 240 total).
  [1/240] ApartmentMidRise_5A_F1_llama-3.3-70b-versatile_seed1542799867: ROUND-TRIP FAILED (1s)  (avg 1s/case, est. remaining 0.1h)
  [2/240] ApartmentMidRise_5A_F1_llama-3.3-70b-versatile_seed1542799869: ROUND-TRIP FAILED (1s)  (avg 1s/case, est. remaining 0.1h)
  [3/240] ApartmentMidRise_5A_F1_llama-3.3-70b-versatile_seed1542799870: ROUND-TRIP FAILED (1s)  (avg 1s/case, est. remaining 0.1h)
  [4/240] ApartmentMidRise_5A_F1_llama-3.3-70b-versatile_seed1542799871: ROUND-TRIP FAILED (1s)  (avg 1s/case, est. remaining 0.0h)
  [5/240] ApartmentMidRise_5A_F1_gpt-oss-120b_seed1542799867: OK (462s)  (avg 93s/case, est. remaining 6.1h)
  [6/240] ApartmentMidRise_5A_F1_llama-3.3-70b-versatile_seed1542799868: OK (685s)  (avg 114s/case, est. remaining 7.4h)
  [7/240] ApartmentMidRise_5A_F1_gpt-oss-120b_seed1542799868: OK (480s)  (avg 135s/case, est. remaining 8.7h)
  [8/240] ApartmentMidRise_5A_F1_gpt-oss-120b_seed1542799869: OK (476s)  (avg 145s/case, est. 

In [ ]:
import pandas as pd
print(len(pd.read_csv(f'{SIM_DIR}/full_batch_log_5A.csv')))

190


In [ ]:
# S4M-G2  Continues the full 5A batch after a runtime restart.
epw_5A = CLIMATE_EPW['5A']

def _run_one_case(args):
    building, climate, source, fmt, model, seed, values, epw_path = args
    tag = f'{building}_{climate}_{source}'
    t0 = time.time()
    out_idf, (rt_passed, rt_report) = build_and_check(building, values, tag)
    if not rt_passed:
        return {'tag': tag, 'building': building, 'climate': climate, 'source': source,
                'format': fmt, 'model': model, 'seed': seed,
                'round_trip_ok': False, 'energyplus_ok': None,
                'duration_s': round(time.time() - t0, 1), 'rt_report': rt_report}
    ep_ok, _ = run_energyplus(out_idf, epw_path, f'{EP_OUT}/{tag}')
    return {'tag': tag, 'building': building, 'climate': climate, 'source': source,
            'format': fmt, 'model': model, 'seed': seed,
            'round_trip_ok': True, 'energyplus_ok': ep_ok,
            'duration_s': round(time.time() - t0, 1), 'rt_report': rt_report}


def _flatten_report(result):
    return [{'tag': result['tag'], 'building': result['building'], 'climate': result['climate'],
             'source': result['source'], 'format': result['format'], 'model': result['model'],
             'seed': result['seed'], 'duration_s': result['duration_s'], **p}
            for p in (result.get('rt_report') or [])]


def run_batch_parallel(manifest_df, epw_path, log_name, adherence_name):
    log_path = f'{SIM_DIR}/{log_name}'
    adherence_path = f'{SIM_DIR}/{adherence_name}'

    log = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []
    adherence_rows = pd.read_csv(adherence_path).to_dict('records') if os.path.exists(adherence_path) else []
    done_tags = {r['tag'] for r in log if r.get('energyplus_ok') is not None or r.get('round_trip_ok') is False}

    print(f'{len(log)} row(s) already in {log_name}, {len(done_tags)} case(s) treated as done, resuming.')

    jobs = []
    for _, row in manifest_df.iterrows():
        tag = f'{row["building"]}_{row["climate"]}_{row["source"]}'
        if tag in done_tags:
            continue
        values = {p: row[p] for p in TARGET_PARAMS}
        jobs.append((row['building'], row['climate'], row['source'],
                     row.get('format'), row.get('model'), row.get('seed'), values, epw_path))

    print(f'{len(jobs)} case(s) remaining (of {len(manifest_df)} total).')
    completed = 0
    batch_start = time.time()
    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(_run_one_case, job): job for job in jobs}
        for future in as_completed(futures):
            result = future.result()
            log.append({k: v for k, v in result.items() if k != 'rt_report'})
            adherence_rows.extend(_flatten_report(result))
            completed += 1
            status = ('OK' if result.get('energyplus_ok')
                       else 'ROUND-TRIP FAILED' if not result['round_trip_ok'] else 'EP FAILED')
            elapsed = time.time() - batch_start
            print(f'  [{completed}/{len(jobs)}] {result["tag"]}: {status} ({result["duration_s"]:.0f}s)  '
                  f'(avg {elapsed/completed:.0f}s/case, est. remaining {elapsed/completed*(len(jobs)-completed)/3600:.1f}h)')
            if completed % 10 == 0:
                pd.DataFrame(log).to_csv(log_path, index=False)
                pd.DataFrame(adherence_rows).to_csv(adherence_path, index=False)

    pd.DataFrame(log).to_csv(log_path, index=False)
    pd.DataFrame(adherence_rows).to_csv(adherence_path, index=False)
    return pd.DataFrame(log)

main_batch_df = run_batch_parallel(sim_manifest_df, epw_5A, 'full_batch_log_5A.csv', 'range_adherence_log_main.csv')
print(f'\n5A main batch: {main_batch_df.energyplus_ok.sum()}/{len(main_batch_df)} successful.')
print(f'Mean duration: {main_batch_df.duration_s.mean():.1f}s/case, '
      f'total: {main_batch_df.duration_s.sum()/3600:.2f}h (sequential-equivalent)')

190 row(s) already in full_batch_log_5A.csv, 190 case(s) treated as done, resuming.
50 case(s) remaining (of 240 total).
  [1/50] SchoolPrimary_5A_F4_llama-3.3-70b-versatile_seed1542799867: OK (490s)  (avg 490s/case, est. remaining 6.7h)
  [2/50] SchoolPrimary_5A_F4_llama-3.3-70b-versatile_seed1542799868: OK (546s)  (avg 273s/case, est. remaining 3.6h)
  [3/50] SchoolPrimary_5A_F4_llama-3.3-70b-versatile_seed1542799869: OK (492s)  (avg 327s/case, est. remaining 4.3h)
  [4/50] SchoolPrimary_5A_F4_llama-3.3-70b-versatile_seed1542799870: OK (498s)  (avg 261s/case, est. remaining 3.3h)
  [5/50] SchoolPrimary_5A_F4_llama-3.3-70b-versatile_seed1542799871: OK (521s)  (avg 301s/case, est. remaining 3.8h)
  [6/50] SchoolPrimary_5A_F4_gpt-oss-120b_seed1542799867: OK (500s)  (avg 257s/case, est. remaining 3.1h)
  [7/50] SchoolPrimary_5A_F4_gpt-oss-120b_seed1542799868: OK (493s)  (avg 285s/case, est. remaining 3.4h)
  [8/50] SchoolPrimary_5A_F4_gpt-oss-120b_seed1542799870: ROUND-TRIP FAILED (1s)  

In [ ]:
#S4M-H — Failure accounting
adherence_df = pd.read_csv(f'{SIM_DIR}/range_adherence_log_main.csv')
n_param_violations = (adherence_df['status'] != 'OK').sum()
print(f'{n_param_violations} parameter-level round-trip issue(s) across {len(adherence_df)} checked values.')
print(adherence_df[adherence_df['status'] != 'OK'].groupby(['format', 'model', 'parameter', 'status']).size().to_string())

def build_failure_log(manifest_df, batch_df, climate_role):
    tags = manifest_df.apply(lambda r: f'{r["building"]}_{r["climate"]}_{r["source"]}', axis=1)
    merged = manifest_df.assign(tag=tags).merge(
        batch_df[['tag', 'round_trip_ok', 'energyplus_ok', 'duration_s']], on='tag', how='left')
    rows = []
    for _, r in merged.iterrows():
        if r['round_trip_ok'] is True and r['energyplus_ok'] is True:
            continue
        status = ('round_trip_failed' if r['round_trip_ok'] is False
                   else 'energyplus_failed' if r['energyplus_ok'] is False else 'not_yet_run')
        rows.append({'building': r['building'], 'climate': r['climate'], 'climate_role': climate_role,
                      'format': r.get('format'), 'model': r.get('model'), 'seed': r.get('seed'),
                      'source': r['source'], 'source_type': r['source_type'], 'status': status,
                      'duration_s': r.get('duration_s')})
    return pd.DataFrame(rows)

main_failures = build_failure_log(sim_manifest_df, main_batch_df, 'main_experiment')
main_failures.to_csv(f'{SIM_DIR}/simulation_failure_log_main.csv', index=False)
print(f'\n{len(main_failures)} non-successful vector(s) out of {len(sim_manifest_df)}.')
if len(main_failures):
    print(main_failures.groupby(['status', 'format', 'model']).size().to_string())

51 parameter-level round-trip issue(s) across 2160 checked values.
format  model                    parameter           status            
F1      llama-3.3-70b-versatile  cooling_setpoint_C  MISSING               24
                                 heating_setpoint_C  MISSING               24
F2      llama-3.3-70b-versatile  outdoor_air         TOLERANCE_EXCEEDED     2
F4      openai/gpt-oss-120b      outdoor_air         TOLERANCE_EXCEEDED     1

27 non-successful vector(s) out of 240.
status             format  model                  
round_trip_failed  F1      llama-3.3-70b-versatile    24
                   F2      llama-3.3-70b-versatile     2
                   F4      openai/gpt-oss-120b         1


In [ ]:
# S4M-I  Final validation + environment record. Reports two distinct
# rates: overall (of all registered vectors) and EnergyPlus-only (of
# vectors that passed round-trip and were actually attempted) -- these
# answer different questions and must not be collapsed into one number.
n_total = len(main_batch_df)
n_rt_rejected = main_batch_df.energyplus_ok.isna().sum()
n_attempted = main_batch_df.energyplus_ok.notna().sum()
n_ok = (main_batch_df.energyplus_ok == True).sum()

overall_rate = n_ok / n_total if n_total else 0
attempted_rate = n_ok / n_attempted if n_attempted else 0

print(f'5A main: {n_ok}/{n_total} overall ({overall_rate*100:.1f}%); '
      f'{n_rt_rejected} round-trip rejected (expected, not a failure); '
      f'{n_ok}/{n_attempted} of EnergyPlus-attempted vectors succeeded ({attempted_rate*100:.1f}%)')
if overall_rate < 0.90:
    print('  WARNING: overall rate below 90% -- check whether this is driven by round-trip '
          'rejections (expected) or genuine EnergyPlus failures (investigate).')
if n_attempted and attempted_rate < 1.0:
    print('  WARNING: at least one EnergyPlus-attempted vector failed the simulation itself -- '
          'inspect simulation_failure_log_main.csv before Sprint 5.')

with open(f'{DATA}/environment_sprint4_main.json', 'w') as f:
    json.dump({'sprint': 'Sprint 4 - Main Experiment', 'python_version': sys.version,
               'platform': platform.platform(), 'energyplus_version': energyplus_version,
               'n_workers': N_WORKERS, 'ep_timeout_s': EP_TIMEOUT_S,
               'master_seed': protocol['master_seed'],
               'n_total_vectors': int(n_total), 'n_round_trip_rejected': int(n_rt_rejected),
               'n_energyplus_attempted': int(n_attempted), 'n_energyplus_ok': int(n_ok)}, f, indent=2)
print('Saved: environment_sprint4_main.json')

5A main: 213/240 overall (88.8%); 27 round-trip rejected (expected, not a failure); 213/213 of EnergyPlus-attempted vectors succeeded (100.0%)
Saved: environment_sprint4_main.json


In [ ]:
# S4M-I-CHECK  Diagnose the rate calculation and confirm the true success rate.
print('energyplus_ok value counts (including None = never attempted):')
print(main_batch_df.energyplus_ok.value_counts(dropna=False).to_string())

print(f'\nsum()  = {main_batch_df.energyplus_ok.sum()}')
print(f'mean() = {main_batch_df.energyplus_ok.mean():.4f}  <- silently divides by non-null count only')
print(f'len()  = {len(main_batch_df)}')

n_round_trip_failed = main_batch_df.energyplus_ok.isna().sum()
n_energyplus_attempted = main_batch_df.energyplus_ok.notna().sum()
n_energyplus_ok = (main_batch_df.energyplus_ok == True).sum()
n_energyplus_failed = (main_batch_df.energyplus_ok == False).sum()

print(f'\nRound-trip rejected (never sent to EnergyPlus): {n_round_trip_failed}')
print(f'EnergyPlus attempted: {n_energyplus_attempted}')
print(f'  -> succeeded: {n_energyplus_ok}')
print(f'  -> failed (EnergyPlus itself crashed): {n_energyplus_failed}')

print(f'\nCorrect overall rate (of all 240 registered vectors): '
      f'{n_energyplus_ok}/{len(main_batch_df)} = {n_energyplus_ok/len(main_batch_df)*100:.1f}%')
print(f'Correct EnergyPlus-only rate (of vectors actually attempted): '
      f'{n_energyplus_ok}/{n_energyplus_attempted} = {n_energyplus_ok/n_energyplus_attempted*100:.1f}%'
      if n_energyplus_attempted else 'N/A')

energyplus_ok value counts (including None = never attempted):
energyplus_ok
True    213
NaN      26
None      1

sum()  = 213
mean() = 1.0000  <- silently divides by non-null count only
len()  = 240

Round-trip rejected (never sent to EnergyPlus): 27
EnergyPlus attempted: 213
  -> succeeded: 213
  -> failed (EnergyPlus itself crashed): 0

Correct overall rate (of all 240 registered vectors): 213/240 = 88.8%
Correct EnergyPlus-only rate (of vectors actually attempted): 213/213 = 100.0%


## Sprint 4 — Ana Batch Doğrulama Notu

Ana deneyde (240 kayıtlı vektör, Buffalo/5A) round-trip doğrulaması
27 vektörü (%11,3) reddetti; kalan 213 vektörün (%88,8) tamamı
EnergyPlus'ta başarıyla tamamlandı.

**27 reddin dökümü:**
- 24 vektör — Llama-3.3-70B, F1 formatı: Sprint 3'ten bilinen Fahrenheit
  kayması (birim verilmeden üretilen setpoint'ler 18–26 °C aralığının
  dışında kaldığı için okunamadı).
- 2 vektör — Llama-3.3-70B, F2 formatı: dış hava debisi round-trip
  toleransını (%2,0) aştı.
- 1 vektör — gpt-oss-120b, F4 formatı: aynı şekilde dış hava debisi
  toleransı aşımı.

**Önemli:** Bu 27 vektör hiçbir zaman EnergyPlus'a gönderilmedi — round-trip
kapısı (Bölüm 3.5, Algoritma 1) tam olarak bunun için var: fiziksel olarak
anlamsız veya hedeflenen değere yazılamamış bir parametrenin simülasyona
girmesini önlemek. Bu nedenle EnergyPlus'ın kendisinin gerçek başarı oranı
**213/213 (\%100)** — motor, kendisine gönderilen hiçbir girdide
başarısız olmamıştır. Genel oran (213/240, \%88,8) ile EnergyPlus-özel oran
(213/213, \%100) bu yüzden makalede **ayrı raporlanmaktadır**; ikisi farklı
soruları yanıtlamaktadır ("kaç vektör nihayetinde simüle edildi" vs.
"simüle edilenlerin kaçı başarılı oldu").

## Sprint 4 — Main Experiment (5A) — Code Block Guide

**S4M-0 — Setup.** Mounts Drive, installs/verifies EnergyPlus 22.1.0 (matches the reference IDFs' native `Version,22.1;`), loads `protocol.json`/`lobo_range_table.json`/`control_parameters.json`, sets per-parameter round-trip tolerances and worker count.

**S4M-A — Extraction.** Comment-anchored IDF parsing (locates a field by its descriptive comment, not its position). Provides `extract_all_params`, used both to read the DOE reference and to verify injected values.

**S4M-B — Injection.** Writes each of the 9 target parameters into every matching object in a building's IDF, using the same zone-area map the extractor uses. Control parameters (infiltration, heating efficiency, roof U-value) are never touched.

**S4M-C — Standard outputs.** Appends `Output:Meter`/`Output:Variable` objects to every generated IDF if not already present, so electricity/gas/heating/cooling output is reported consistently across all cases.

**S4M-D — Round-trip + EnergyPlus runner.** `round_trip_check` re-reads an injected IDF and compares to target; `run_energyplus` runs the engine with a 1800s timeout, caught rather than left to crash the caller.

**S4M-E — Manifest.** Builds `simulation_vector_manifest.csv`: 200 LLM outputs + 10 deterministic baselines + 25 a priori uniform-random draws + 5 DOE-reference vectors = 240 vectors, 5A only.

**S4M-F — Smoke test.** Runs 28 representative cases (DOE reference, one per format×model, one per baseline, one per random draw). A round-trip rejection (e.g. Llama's F1 Fahrenheit drift) is excluded from the pass/fail gate; only EnergyPlus failures among round-trip-passing cases block progress. Writes `smoke_test_results_main.csv`.

**S4M-G — Full batch.** Runs all 240 vectors in parallel (checkpointed every 10 completions). Writes `full_batch_log_5A.csv` (per-case outcome + duration) and `range_adherence_log_main.csv` (per-parameter target/readback/diff_pct/status for every case, not just the smoke sample).

**S4M-H — Failure accounting.** Summarizes parameter-level round-trip issues from `range_adherence_log_main.csv`; writes `simulation_failure_log_main.csv` (Building | Climate | Format | Seed | Status).

**S4M-I — Final validation.** Reports overall success rate (warns below 90%); writes `environment_sprint4_main.json` (authoritative EnergyPlus version/settings record for this experiment).

### Output files

| File | Contents |
|---|---|
| `simulation_vector_manifest.csv` | All 240 registered vectors and their 9 parameter values |
| `smoke_test_results_main.csv` | 28-case pilot run: round-trip/EnergyPlus status per case |
| `full_batch_log_5A.csv` | Per-vector outcome (round_trip_ok, energyplus_ok, duration_s) |
| `range_adherence_log_main.csv` | Per-parameter target vs. readback vs. tolerance, every vector |
| `simulation_failure_log_main.csv` | Every non-successful vector with reason |
| `environment_sprint4_main.json` | EnergyPlus version, worker count, timeout, master seed |
| `idf_generated/*.idf` | Every generated IDF (one per vector) |
| `energyplus_runs/<tag>/` | Raw EnergyPlus output per vector (`eplusout.err`, `eplusmtr.csv`, etc.) |

---

## Sprint 4 — Ana Deney (5A) — Kod Bloğu Rehberi

**S4M-0 — Kurulum.** Drive'ı bağlar, EnergyPlus 22.1.0'ı kurar/doğrular (referans IDF'lerin native `Version,22.1;` etiketiyle eşleşiyor), `protocol.json`/`lobo_range_table.json`/`control_parameters.json`'ı yükler, parametre-özel round-trip toleranslarını ve işçi sayısını ayarlar.

**S4M-A — Çıkarım.** Yorum-çapalı IDF ayrıştırma (bir alanı konumuna göre değil, açıklayıcı yorumuna göre bulur). Hem DOE referansını okumak hem de enjekte edilen değerleri doğrulamak için kullanılan `extract_all_params`'ı sağlar.

**S4M-B — Enjeksiyon.** 9 hedef parametrenin her birini, çıkarıcının kullandığı aynı zon-alan haritasıyla, binanın IDF'sindeki her eşleşen nesneye yazar. Kontrol parametrelerine (infiltrasyon, ısıtma verimi, çatı U-değeri) hiç dokunulmaz.

**S4M-C — Standart çıktılar.** Her üretilen IDF'ye, yoksa `Output:Meter`/`Output:Variable` nesnelerini ekler — elektrik/gaz/ısıtma/soğutma çıktısının tüm vakalarda tutarlı raporlanmasını sağlar.

**S4M-D — Round-trip + EnergyPlus çalıştırıcı.** `round_trip_check` enjekte edilmiş IDF'yi tekrar okuyup hedefle karşılaştırır; `run_energyplus` motoru 1800s zaman aşımıyla çalıştırır, zaman aşımı yakalanır, çağıranı çökertmez.

**S4M-E — Manifest.** `simulation_vector_manifest.csv`'yi üretir: 200 LLM çıktısı + 10 deterministik baseline + 25 önceden-sabitlenmiş uniform-rastgele çekiliş + 5 DOE-referans vektörü = 240 vektör, yalnızca 5A.

**S4M-F — Smoke test.** 28 temsili vakayı çalıştırır (DOE referansı, format×model başına bir, baseline başına bir, rastgele çekiliş başına bir). Round-trip reddi (ör. Llama'nın F1 Fahrenheit kayması) geçti/kaldı kararından hariç tutulur; yalnızca round-trip'i geçen vakalar arasındaki EnergyPlus hataları ilerlemeyi engeller. `smoke_test_results_main.csv`'yi yazar.

**S4M-G — Tam batch.** 240 vektörün tamamını paralel çalıştırır (her 10 tamamlanmada bir checkpoint). `full_batch_log_5A.csv` (vaka başına sonuç + süre) ve `range_adherence_log_main.csv`'yi (yalnızca smoke test'te değil, her vakada parametre başına hedef/geri-okuma/fark/durum) yazar.

**S4M-H — Hata dökümü.** `range_adherence_log_main.csv`'den parametre düzeyinde round-trip sorunlarını özetler; `simulation_failure_log_main.csv`'yi (Bina | İklim | Format | Tohum | Durum) yazar.

**S4M-I — Son doğrulama.** Genel başarı oranını raporlar (%90 altında uyarır); `environment_sprint4_main.json`'ı (bu deney için asıl EnergyPlus sürümü/ayar kaydı) yazar.

### Çıktı dosyaları

| Dosya | İçerik |
|---|---|
| `simulation_vector_manifest.csv` | Kayıtlı 240 vektörün tamamı ve 9 parametre değeri |
| `smoke_test_results_main.csv` | 28 vakalık pilot koşu: vaka başına round-trip/EnergyPlus durumu |
| `full_batch_log_5A.csv` | Vektör başına sonuç (round_trip_ok, energyplus_ok, duration_s) |
| `range_adherence_log_main.csv` | Her vektörde parametre başına hedef vs. geri-okuma vs. tolerans |
| `simulation_failure_log_main.csv` | Başarısız her vektör, nedeniyle birlikte |
| `environment_sprint4_main.json` | EnergyPlus sürümü, işçi sayısı, zaman aşımı, master seed |
| `idf_generated/*.idf` | Üretilen her IDF (vektör başına bir) |
| `energyplus_runs/<tag>/` | Vektör başına ham EnergyPlus çıktısı (`eplusout.err`, `eplusmtr.csv` vb.) |